In [34]:
# ==============================================================================
# TAHAP 1: MEMBACA DATA
# ==============================================================================
from google.colab import drive
import pandas as pd
import numpy as np

print("="*50)
print("="*50)

# Link diubah menjadi format 'uc?id=' agar bisa langsung dibaca Pandas
url_id = '1aDEAjdyTd8qtB-AMRmAYXzxU08jNPcY2'
file_path = f'https://drive.google.com/uc?id={url_id}'

try:
    df = pd.read_csv(file_path)
    print("\n✅ Data berhasil di-load dari Drive!")
except FileNotFoundError:
    print("\n❌ ERROR: File tidak ditemukan. Cek folder Drive kamu.")


✅ Data berhasil di-load dari Drive!


In [ ]:
# ==============================================================================
# TAHAP 2: MENGHITUNG PRIOR P(C)
# ==============================================================================
target_col = 'Outcome'
kelas_target = sorted(df[target_col].unique())
total_data = len(df)
prior_probs = {}

print("\n" + "="*50)
print("="*50)
for kelas in kelas_target:
    jumlah_kelas = len(df[df[target_col] == kelas])
    prior_probs[kelas] = jumlah_kelas / total_data
    print(f"P(Outcome={kelas}) = {jumlah_kelas} / {total_data} = {prior_probs[kelas]:.4f}")


TAHAP 2: MENGHITUNG PRIOR P(C)
P(Outcome=0) = 500 / 768 = 0.6510
P(Outcome=1) = 268 / 768 = 0.3490


In [ ]:
# ==============================================================================
# TAHAP 3: MENGHITUNG & MENAMPILKAN MEAN (μ) DAN VARIANCE (σ²)
# ==============================================================================
fitur_cols = [col for col in df.columns if col != target_col]
mean_dict = {0: {}, 1: {}}
var_dict = {0: {}, 1: {}}

print("\n" + "="*50)
print("="*50)
print(f"{'Fitur':<25} | {'Kelas':<6} | {'Mean (μ)':<10} | {'Variance (σ²)':<10}")
print("-" * 65)

for fitur in fitur_cols:
    for kelas in kelas_target:
        data_fitur = df[df[target_col] == kelas][fitur]
        m = data_fitur.mean()
        v = data_fitur.var(ddof=1)
        mean_dict[kelas][fitur] = m
        var_dict[kelas][fitur] = v
        print(f"{fitur:<25} | {kelas:<6} | {m:<10.4f} | {v:<10.4f}")


TAHAP 3: NILAI MEAN (μ) DAN VARIANCE (σ²) TIAP FITUR
Fitur                     | Kelas  | Mean (μ)   | Variance (σ²)
-----------------------------------------------------------------
Pregnancies               | 0      | 3.2980     | 9.1034    
Pregnancies               | 1      | 4.8657     | 13.9969   
Glucose                   | 0      | 109.9800   | 683.3623  
Glucose                   | 1      | 141.2575   | 1020.1395 
BloodPressure             | 0      | 68.1840    | 326.2747  
BloodPressure             | 1      | 70.8246    | 461.8980  
SkinThickness             | 0      | 19.6640    | 221.7105  
SkinThickness             | 1      | 22.1642    | 312.5722  
Insulin                   | 0      | 68.7920    | 9774.3454 
Insulin                   | 1      | 100.3358   | 19234.6733
BMI                       | 0      | 30.3042    | 59.1339   
BMI                       | 1      | 35.1425    | 52.7507   
DiabetesPedigreeFunction  | 0      | 0.4297     | 0.0895    
DiabetesPedigreeFunctio

In [ ]:
# ==============================================================================
# TAHAP 4: DATA PASIEN BARU & PERHITUNGAN RUMUS LIKELIHOOD
# ==============================================================================
pasien_baru = {
    'Pregnancies': 2, 'Glucose': 130, 'BloodPressure': 70,
    'SkinThickness': 25, 'Insulin': 100, 'BMI': 28.5,
    'DiabetesPedigreeFunction': 0.45, 'Age': 33
}

print("\n" + "="*50)
print("="*50)

def hitung_gaussian(x, mean, var):
    depan = 1 / np.sqrt(2 * np.pi * var)
    eksponen = np.exp(-((x - mean)**2) / (2 * var))
    return depan * eksponen

likelihoods = {0: {}, 1: {}}

for kelas in kelas_target:
    print(f"\n--- [OUTCOME {kelas}] ---")
    for fitur in fitur_cols:
        x_i = pasien_baru[fitur]
        mu = mean_dict[kelas][fitur]
        sigma2 = var_dict[kelas][fitur]
        prob = hitung_gaussian(x_i, mu, sigma2)
        likelihoods[kelas][fitur] = prob
        print(f"P({fitur}={x_i}|{kelas}) = (1/sqrt(2*3.14*{sigma2:.2f})) * exp(-({x_i}-{mu:.2f})^2 / (2*{sigma2:.2f})) = {prob:.6e}")


TAHAP 4: HITUNG LIKELIHOOD DENGAN RUMUS GAUSSIAN

--- [OUTCOME 0] ---
P(Pregnancies=2|0) = (1/sqrt(2*3.14*9.10)) * exp(-(2-3.30)^2 / (2*9.10)) = 1.205369e-01
P(Glucose=130|0) = (1/sqrt(2*3.14*683.36)) * exp(-(130-109.98)^2 / (2*683.36)) = 1.138217e-02
P(BloodPressure=70|0) = (1/sqrt(2*3.14*326.27)) * exp(-(70-68.18)^2 / (2*326.27)) = 2.197473e-02
P(SkinThickness=25|0) = (1/sqrt(2*3.14*221.71)) * exp(-(25-19.66)^2 / (2*221.71)) = 2.512639e-02
P(Insulin=100|0) = (1/sqrt(2*3.14*9774.35)) * exp(-(100-68.79)^2 / (2*9774.35)) = 3.839098e-03
P(BMI=28.5|0) = (1/sqrt(2*3.14*59.13)) * exp(-(28.5-30.30)^2 / (2*59.13)) = 5.047062e-02
P(DiabetesPedigreeFunction=0.45|0) = (1/sqrt(2*3.14*0.09)) * exp(-(0.45-0.43)^2 / (2*0.09)) = 1.330816e+00
P(Age=33|0) = (1/sqrt(2*3.14*136.13)) * exp(-(33-31.19)^2 / (2*136.13)) = 3.378320e-02

--- [OUTCOME 1] ---
P(Pregnancies=2|1) = (1/sqrt(2*3.14*14.00)) * exp(-(2-4.87)^2 / (2*14.00)) = 7.952297e-02
P(Glucose=130|1) = (1/sqrt(2*3.14*1020.14)) * exp(-(130-141.26)^

In [35]:
# ==============================================================================
# TAHAP 5: MENGHITUNG SKOR AKHIR (POSTERIOR) BERDASARKAN RUMUS
# ==============================================================================
print("="*50)
print("="*50)
print("Rumus Naive Bayes: P(C|X) = P(C) * P(X1|C) * P(X2|C) * ... * P(Xn|C)\n")

skor_akhir = {}

for kelas in kelas_target:
    skor = prior_probs[kelas]
    print(f"Skor Outcome {kelas} = Prior P(Outcome={kelas})", end="")

    for fitur in fitur_cols:
        print(f" * P({fitur}|{kelas})", end="")
        skor *= likelihoods[kelas][fitur]

    skor_akhir[kelas] = skor
    print(f"\nSkor Outcome {kelas} = {prior_probs[kelas]:.4f}", end="")
    for fitur in fitur_cols:
         print(f" * {likelihoods[kelas][fitur]:.2e}", end="")

    print(f"\nTotal Skor Outcome {kelas} = {skor:.10e}\n")

print("="*50)
print("KESIMPULAN AKHIR")
print("="*50)
if skor_akhir[1] > skor_akhir[0]:
    print(f"Karena Total Skor Outcome 1 ({skor_akhir[1]:.10e}) > Skor Outcome 0 ({skor_akhir[0]:.10e})")
    print("Maka prediksi akhirnya adalah: KELAS 1 (TERKENA DIABETES)")
else:
    print(f"Karena Total Skor Outcome 0 ({skor_akhir[0]:.10e}) > Skor Outcome 1 ({skor_akhir[1]:.10e})")
    print("Maka prediksi akhirnya adalah: KELAS 0 (TIDAK DIABETES)")

Rumus Naive Bayes: P(C|X) = P(C) * P(X1|C) * P(X2|C) * ... * P(Xn|C)

Skor Outcome 0 = Prior P(Outcome=0) * P(Pregnancies|0) * P(Glucose|0) * P(BloodPressure|0) * P(SkinThickness|0) * P(Insulin|0) * P(BMI|0) * P(DiabetesPedigreeFunction|0) * P(Age|0)
Skor Outcome 0 = 0.6510 * 1.21e-01 * 1.14e-02 * 2.20e-02 * 2.51e-02 * 3.84e-03 * 5.05e-02 * 1.33e+00 * 3.38e-02
Total Skor Outcome 0 = 4.2962921123e-12

Skor Outcome 1 = Prior P(Outcome=1) * P(Pregnancies|1) * P(Glucose|1) * P(BloodPressure|1) * P(SkinThickness|1) * P(Insulin|1) * P(BMI|1) * P(DiabetesPedigreeFunction|1) * P(Age|1)
Skor Outcome 1 = 0.3490 * 7.95e-02 * 1.17e-02 * 1.85e-02 * 2.23e-02 * 2.88e-03 * 3.62e-02 * 1.03e+00 * 3.40e-02
Total Skor Outcome 1 = 4.9103765606e-13

KESIMPULAN AKHIR
Karena Total Skor Outcome 0 (4.2962921123e-12) > Skor Outcome 1 (4.9103765606e-13)
Maka prediksi akhirnya adalah: KELAS 0 (TIDAK DIABETES)
